In [2]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
import plotly.express as px
import os
import warnings
warnings.filterwarnings('ignore')
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.filterwarnings('ignore', category=ValueWarning)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import sys
sys.path.append(os.path.abspath('..'))

from scripts.data_loader import MacroDataLoader
from scripts.validator import WalkForwardValidator
from scripts.evaluator import ModelEvaluator

# Load feature matrix

loader = MacroDataLoader('../data/master_data1.csv')
loader.load()
loader.build_lag_features()
X, y = loader.get_feature_matrix()

X_train = X[X.index < '2022-01-01']
X_test = X[X.index >= '2022-01-01']
y_train = y[y.index < '2022-01-01']
y_test = y[y.index >= '2022-01-01']
print(X_train.shape, X_test.shape)

(108, 6) (52, 6)


In [6]:
# Elastic Net has two parameters

params_en = {
    'alpha': [0.001, 0.01, 0.1, 1.0],    # alpha controls overall regularization strength
    'l1_ratio': [0.2, 0.5, 0.8, 1.0]     # l1_ratio controls LASSO vs Ridge balance — 0 is pure Ridge, 1 is pure LASSO
}

tscv = TimeSeriesSplit(n_splits=5)

grid_en = GridSearchCV(ElasticNet(max_iter=10000), params_en,
                       cv=tscv, scoring='neg_root_mean_squared_error',
                       n_jobs=-1)

grid_en.fit(X_train, y_train)

print("Best params:", grid_en.best_params_)
print("Best CV RMSE:", round(-grid_en.best_score_, 4))

Best params: {'alpha': 0.01, 'l1_ratio': 1.0}
Best CV RMSE: 0.8433


In [7]:
# Walk forward validation
model_en = ElasticNet(**grid_en.best_params_, max_iter=10000)
validator_en = WalkForwardValidator(model_en, X, y, test_start='2022-01-01')
validator_en.run()
print(validator_en.get_metrics())

{'rmse': np.float64(0.7213), 'mae': 0.5458}


## Notebook 05 Summary — Elastic Net Results

**Model:** Elastic Net with walk forward validation
**Best params:** alpha 0.01, l1_ratio 1.0 (pure LASSO)
**RMSE:** 0.7213 | **MAE:** 0.5458

**Key Finding**
ElasticNet performance, between ARIMA and XGBoost is better than 
XGBoost standalone but cannot beat pure autoregression.
l1_ratio of 1.0 means LASSO won — most features zeroed out,
only CPI lags dominate. Consistent with ARIMA finding that
inflation persistence is the strongest predictive signal.

**Note**
All model comparison is in notebook 07.